# core

> Fill in a module description here

In [ ]:
#| default_exp core

My goal is to refactor this module: https://raw.githubusercontent.com/franckalbinet/evaluatr/refs/heads/main/nbs/03_ocr.ipynb but as a single package with no dependency on `evaluatr`. The idea is to pass a pdf file or a folder containing multiple ones and converting them using batch ocr. 

Can we go through this task together. IMPORTANT: I'd like to leverage fastcore and the fastai coding style as much as possible.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! I'd be happy to help you refactor this OCR module. Let me first take a look at the notebook to understand what we're working with.

<details class='tool-usage-details'>

```json
{
  "id": "toolu_01CfE7FmjUnLnk2boWeGdVV1",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://raw.githubusercontent.com/franckalbinet/evaluatr/refs/heads/main/nbs/03_ocr.ipynb",
      "as_md": "False"
    }
  },
  "result": "'{\\n \"cells\": [\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"# OCR\\\\n\",\\n    \"\\\\n\",\\n    \"> OCR evaluation reports\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| default_exp ocr\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| export\\\\n\",\\n    \"import os\\\\n\",\\n    \"import re\\\\n\",\\n    \"from io import BytesIO\\\\n\",\\n    \"from dotenv import load_dotenv\\\\n\",\\n    \"import base64\\\\n\",\\n    \"from fastcore.all import *\\\\n\",\\n    \"from rich import print\\\\n\",\\n    \"import urllib.parse\\\\n\",\\n    \"from pathlib import Path\\\\n\",\\n    \"from tqdm import tqdm\\\\n\",\\n    \"import json\\\\n\",\\n    \"import logging\\\\n\",\\n    \"import tempfile\\\\n\",\\n    \"from PIL import Image\\\\n\",\\n    \"import time\\\\n\",\\n    \"\\\\n\",\\n    \"from mistralai import Mistral\\\\n\",\\n    \"from evaluatr.readers import load_evals\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| export\\\\n\",\\n    \"load_dotenv()\\\\n\",\\n    \"mistral_api_key = os.getenv(\\\\\"MISTRAL_API_KEY\\\\\")\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| export\\\\n\",\\n    \"src_dir = Path(\\\\\"../_data/\\\\\")\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"For testing purposes, let\\'s load evaluations:\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [\\n    {\\n     \"data\": {\\n      \"text/html\": [\\n       \"<pre style=\\\\\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,\\'DejaVu Sans Mono\\',consolas,\\'Courier New\\',monospace\\\\\"><span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"    <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'id\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'1a57974ab89d7280988aa6b706147ce1\\'</span>,\\\\n\",\\n       \"    <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'docs\\'</span>: <span style=\\\\\"font-weight: bold\\\\\">[</span>\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Document Subtype\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation report\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File URL\\'</span>: \\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">INAL_Abderrahim%20EL%20MOULAT.pdf\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File description\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Report\\'</span>\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">}</span>,\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Document Subtype\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation brief\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File URL\\'</span>: \\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abde</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">rrahim%20EL%20MOULAT.pdf\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File description\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Brief\\'</span>\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">}</span>\\\\n\",\\n       \"    <span style=\\\\\"font-weight: bold\\\\\">]</span>,\\\\n\",\\n       \"    <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'meta\\'</span>: <span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Title\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'EX-POST EVALUATION OF THE PROJECT:  NIGERIA: STRENGTHENING REINTEGRATION FOR RETURNEES (SRARP)  -</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">PHASE II\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Year\\'</span>: <span style=\\\\\"color: #008080; text-decoration-color: #008080; font-weight: bold\\\\\">2023</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Author\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Abderrahim El Moulat\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Best Practicesor Lessons Learnt\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Date of Publication\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'2023-05-10\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Donor\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Government of Germany\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Brief\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Commissioner\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Donor, IOM\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Coverage\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Country\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Period From Date\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'nan\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Period To Date\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'NaT\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Executive Summary\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'External Version of the Report\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'No\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Languages\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'English\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Migration Thematic Areas\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Assistance to vulnerable migrants, Migrant training and integration (including</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">community cohesion), Migration health (assessment, travel, health promotion, crisis-affected), Return and AVRR\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Name of Project(s) Being Evaluated\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Number of Pages Excluding annexes\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Other Documents Included\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Project Code\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'RR.0163\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Countries Covered\\'</span>: <span style=\\\\\"font-weight: bold\\\\\">[</span><span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Nigeria\\'</span><span style=\\\\\"font-weight: bold\\\\\">]</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Regions Covered\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'RO Dakar\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Relevant Crosscutting Themes\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Gender, Rights-based approach\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Report Published\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Terms of Reference\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'No\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Type of Evaluation Scope\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Programme/Project\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Type of Evaluation Timing\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Ex-post (after the end of the project/programme)\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Type of Evaluator\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Internal\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Level of Evaluation\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Decentralized\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Document Subtype\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation report, Evaluation brief\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File URL\\'</span>: \\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">INAL_Abderrahim%20EL%20MOULAT.pdf,   </span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abder</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">rahim%20EL%20MOULAT.pdf\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File description\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Report , Evaluation Brief\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Management response\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'No\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Date added\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Fri, 07/07/2023 - 15:35\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Metaevaluation\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'2020-24\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'exclude\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'reason\\'</span>: nan\\\\n\",\\n       \"    <span style=\\\\\"font-weight: bold\\\\\">}</span>\\\\n\",\\n       \"<span style=\\\\\"font-weight: bold\\\\\">}</span>\\\\n\",\\n       \"</pre>\\\\n\"\\n      ],\\n      \"text/plain\": [\\n       \"\\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"    \\\\u001b[32m\\'id\\'\\\\u001b[0m: \\\\u001b[32m\\'1a57974ab89d7280988aa6b706147ce1\\'\\\\u001b[0m,\\\\n\",\\n       \"    \\\\u001b[32m\\'docs\\'\\\\u001b[0m: \\\\u001b[1m[\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"            \\\\u001b[32m\\'Document Subtype\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation report\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File URL\\'\\\\u001b[0m: \\\\n\",\\n       \"\\\\u001b[32m\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mINAL_Abderrahim%20EL%20MOULAT.pdf\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File description\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation Report\\'\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[1m}\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"            \\\\u001b[32m\\'Document Subtype\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation brief\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File URL\\'\\\\u001b[0m: \\\\n\",\\n       \"\\\\u001b[32m\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abde\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mrrahim%20EL%20MOULAT.pdf\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File description\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation Brief\\'\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[1m}\\\\u001b[0m\\\\n\",\\n       \"    \\\\u001b[1m]\\\\u001b[0m,\\\\n\",\\n       \"    \\\\u001b[32m\\'meta\\'\\\\u001b[0m: \\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[32m\\'Title\\'\\\\u001b[0m: \\\\u001b[32m\\'EX-POST EVALUATION OF THE PROJECT:  NIGERIA: STRENGTHENING REINTEGRATION FOR RETURNEES \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32mSRARP\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m  -\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mPHASE II\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Year\\'\\\\u001b[0m: \\\\u001b[1;36m2023\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Author\\'\\\\u001b[0m: \\\\u001b[32m\\'Abderrahim El Moulat\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Best Practicesor Lessons Learnt\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Date of Publication\\'\\\\u001b[0m: \\\\u001b[32m\\'2023-05-10\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Donor\\'\\\\u001b[0m: \\\\u001b[32m\\'Government of Germany\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Brief\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Commissioner\\'\\\\u001b[0m: \\\\u001b[32m\\'Donor, IOM\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Coverage\\'\\\\u001b[0m: \\\\u001b[32m\\'Country\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Period From Date\\'\\\\u001b[0m: \\\\u001b[32m\\'nan\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Period To Date\\'\\\\u001b[0m: \\\\u001b[32m\\'NaT\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Executive Summary\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'External Version of the Report\\'\\\\u001b[0m: \\\\u001b[32m\\'No\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Languages\\'\\\\u001b[0m: \\\\u001b[32m\\'English\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Migration Thematic Areas\\'\\\\u001b[0m: \\\\u001b[32m\\'Assistance to vulnerable migrants, Migrant training and integration \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32mincluding\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mcommunity cohesion\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m, Migration health \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32massessment, travel, health promotion, crisis-affected\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m, Return and AVRR\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Name of Project\\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32ms\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m Being Evaluated\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'Number of Pages Excluding annexes\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'Other Documents Included\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'Project Code\\'\\\\u001b[0m: \\\\u001b[32m\\'RR.0163\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Countries Covered\\'\\\\u001b[0m: \\\\u001b[1m[\\\\u001b[0m\\\\u001b[32m\\'Nigeria\\'\\\\u001b[0m\\\\u001b[1m]\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Regions Covered\\'\\\\u001b[0m: \\\\u001b[32m\\'RO Dakar\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Relevant Crosscutting Themes\\'\\\\u001b[0m: \\\\u001b[32m\\'Gender, Rights-based approach\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Report Published\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Terms of Reference\\'\\\\u001b[0m: \\\\u001b[32m\\'No\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Type of Evaluation Scope\\'\\\\u001b[0m: \\\\u001b[32m\\'Programme/Project\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Type of Evaluation Timing\\'\\\\u001b[0m: \\\\u001b[32m\\'Ex-post \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32mafter the end of the project/programme\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Type of Evaluator\\'\\\\u001b[0m: \\\\u001b[32m\\'Internal\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Level of Evaluation\\'\\\\u001b[0m: \\\\u001b[32m\\'Decentralized\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Document Subtype\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation report, Evaluation brief\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'File URL\\'\\\\u001b[0m: \\\\n\",\\n       \"\\\\u001b[32m\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mINAL_Abderrahim%20EL%20MOULAT.pdf,   \\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mhttps://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abder\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mrahim%20EL%20MOULAT.pdf\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'File description\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation Report , Evaluation Brief\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Management response\\'\\\\u001b[0m: \\\\u001b[32m\\'No\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Date added\\'\\\\u001b[0m: \\\\u001b[32m\\'Fri, 07/07/2023 - 15:35\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Metaevaluation\\'\\\\u001b[0m: \\\\u001b[32m\\'2020-24\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'exclude\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'reason\\'\\\\u001b[0m: nan\\\\n\",\\n       \"    \\\\u001b[1m}\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[1m}\\\\u001b[0m\\\\n\"\\n      ]\\n     },\\n     \"metadata\": {},\\n     \"output_type\": \"display_data\"\\n    }\\n   ],\\n   \"source\": [\\n    \"#| eval: false\\\\n\",\\n    \"fname_json = \\'../_data/output/evaluations.json\\'\\\\n\",\\n    \"evals = load_evals(fname_json)\\\\n\",\\n    \"print(evals[0])\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"To find a particular evaluation per name (e.g \\\\\"Final Evaluation of the EU-IOM Joint ...\\\\\"), we can simply:\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [\\n    {\\n     \"data\": {\\n      \"text/plain\": [\\n       \"[{\\'id\\': \\'49d2fba781b6a7c0d94577479636ee6f\\',\\\\n\",\\n       \"  \\'docs\\': [{\\'Document Subtype\\': \\'Evaluation report\\',\\\\n\",\\n       \"    \\'File URL\\': \\'https://ev<TRUNCATED>"
}
```

</details>

Excellent! I can see this is a notebook for OCR processing of evaluation PDFs using Mistral's API. Let me understand your goal:

You want to refactor this into a standalone package that:
- Has no dependency on `evaluatr` 
- Can handle a single PDF file or a folder with multiple PDFs
- Uses batch OCR processing
- Follows fastcore/fastai coding style

Before we start refactoring, I have a few questions to help me understand your needs better:

1. Do you want to keep the evaluation-specific structure (with eval IDs and document subtypes), or simplify it to just process any PDF(s) to markdown?

2. Should the package still use Mistral's API for OCR, or are you open to other OCR solutions?

3. What would you like the main API to look like? For example, something like `ocr_pdf(path)` or `batch_ocr(folder)`?

1. no I want to be independent from evaluation-specific sructure and should be available for any pdfs to markdown
2. no we use mistral batch ocr only
3. yes `ocr_pdf` make sense

All default naming of folder hosting the converted pdf to markdown, image folders, ... should be configurable so that we can reproduce the current implementation as well as adapt it to custom needs.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! So we're building a clean, general-purpose PDF-to-markdown OCR package using Mistral's batch API, with configurable paths.

Let me look at the key functions in the notebook to understand the current implementation better:

<details class='tool-usage-details'>

```json
{
  "id": "toolu_01BZvPBriAfTsda46iC68msJ",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://raw.githubusercontent.com/franckalbinet/evaluatr/refs/heads/main/nbs/03_ocr.ipynb",
      "as_md": "True"
    }
  },
  "result": "'{\\n \"cells\": [\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"# OCR\\\\n\",\\n    \"\\\\n\",\\n    \"> OCR evaluation reports\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| default_exp ocr\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| export\\\\n\",\\n    \"import os\\\\n\",\\n    \"import re\\\\n\",\\n    \"from io import BytesIO\\\\n\",\\n    \"from dotenv import load_dotenv\\\\n\",\\n    \"import base64\\\\n\",\\n    \"from fastcore.all import *\\\\n\",\\n    \"from rich import print\\\\n\",\\n    \"import urllib.parse\\\\n\",\\n    \"from pathlib import Path\\\\n\",\\n    \"from tqdm import tqdm\\\\n\",\\n    \"import json\\\\n\",\\n    \"import logging\\\\n\",\\n    \"import tempfile\\\\n\",\\n    \"from PIL import Image\\\\n\",\\n    \"import time\\\\n\",\\n    \"\\\\n\",\\n    \"from mistralai import Mistral\\\\n\",\\n    \"from evaluatr.readers import load_evals\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| export\\\\n\",\\n    \"load_dotenv()\\\\n\",\\n    \"mistral_api_key = os.getenv(\\\\\"MISTRAL_API_KEY\\\\\")\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| export\\\\n\",\\n    \"src_dir = Path(\\\\\"../_data/\\\\\")\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"For testing purposes, let\\'s load evaluations:\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [\\n    {\\n     \"data\": {\\n      \"text/html\": [\\n       \"<pre style=\\\\\"white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,\\'DejaVu Sans Mono\\',consolas,\\'Courier New\\',monospace\\\\\"><span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"    <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'id\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'1a57974ab89d7280988aa6b706147ce1\\'</span>,\\\\n\",\\n       \"    <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'docs\\'</span>: <span style=\\\\\"font-weight: bold\\\\\">[</span>\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Document Subtype\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation report\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File URL\\'</span>: \\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">INAL_Abderrahim%20EL%20MOULAT.pdf\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File description\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Report\\'</span>\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">}</span>,\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Document Subtype\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation brief\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File URL\\'</span>: \\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abde</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">rrahim%20EL%20MOULAT.pdf\\'</span>,\\\\n\",\\n       \"            <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File description\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Brief\\'</span>\\\\n\",\\n       \"        <span style=\\\\\"font-weight: bold\\\\\">}</span>\\\\n\",\\n       \"    <span style=\\\\\"font-weight: bold\\\\\">]</span>,\\\\n\",\\n       \"    <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'meta\\'</span>: <span style=\\\\\"font-weight: bold\\\\\">{</span>\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Title\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'EX-POST EVALUATION OF THE PROJECT:  NIGERIA: STRENGTHENING REINTEGRATION FOR RETURNEES (SRARP)  -</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">PHASE II\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Year\\'</span>: <span style=\\\\\"color: #008080; text-decoration-color: #008080; font-weight: bold\\\\\">2023</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Author\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Abderrahim El Moulat\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Best Practicesor Lessons Learnt\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Date of Publication\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'2023-05-10\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Donor\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Government of Germany\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Brief\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Commissioner\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Donor, IOM\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Coverage\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Country\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Period From Date\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'nan\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Period To Date\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'NaT\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Executive Summary\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'External Version of the Report\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'No\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Languages\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'English\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Migration Thematic Areas\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Assistance to vulnerable migrants, Migrant training and integration (including</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">community cohesion), Migration health (assessment, travel, health promotion, crisis-affected), Return and AVRR\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Name of Project(s) Being Evaluated\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Number of Pages Excluding annexes\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Other Documents Included\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Project Code\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'RR.0163\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Countries Covered\\'</span>: <span style=\\\\\"font-weight: bold\\\\\">[</span><span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Nigeria\\'</span><span style=\\\\\"font-weight: bold\\\\\">]</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Regions Covered\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'RO Dakar\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Relevant Crosscutting Themes\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Gender, Rights-based approach\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Report Published\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Yes\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Terms of Reference\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'No\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Type of Evaluation Scope\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Programme/Project\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Type of Evaluation Timing\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Ex-post (after the end of the project/programme)\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Type of Evaluator\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Internal\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Level of Evaluation\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Decentralized\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Document Subtype\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation report, Evaluation brief\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File URL\\'</span>: \\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">INAL_Abderrahim%20EL%20MOULAT.pdf,   </span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abder</span>\\\\n\",\\n       \"<span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">rahim%20EL%20MOULAT.pdf\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'File description\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Evaluation Report , Evaluation Brief\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Management response\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'No\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Date added\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Fri, 07/07/2023 - 15:35\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'Metaevaluation\\'</span>: <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'2020-24\\'</span>,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'exclude\\'</span>: nan,\\\\n\",\\n       \"        <span style=\\\\\"color: #008000; text-decoration-color: #008000\\\\\">\\'reason\\'</span>: nan\\\\n\",\\n       \"    <span style=\\\\\"font-weight: bold\\\\\">}</span>\\\\n\",\\n       \"<span style=\\\\\"font-weight: bold\\\\\">}</span>\\\\n\",\\n       \"</pre>\\\\n\"\\n      ],\\n      \"text/plain\": [\\n       \"\\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"    \\\\u001b[32m\\'id\\'\\\\u001b[0m: \\\\u001b[32m\\'1a57974ab89d7280988aa6b706147ce1\\'\\\\u001b[0m,\\\\n\",\\n       \"    \\\\u001b[32m\\'docs\\'\\\\u001b[0m: \\\\u001b[1m[\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"            \\\\u001b[32m\\'Document Subtype\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation report\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File URL\\'\\\\u001b[0m: \\\\n\",\\n       \"\\\\u001b[32m\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mINAL_Abderrahim%20EL%20MOULAT.pdf\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File description\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation Report\\'\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[1m}\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"            \\\\u001b[32m\\'Document Subtype\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation brief\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File URL\\'\\\\u001b[0m: \\\\n\",\\n       \"\\\\u001b[32m\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abde\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mrrahim%20EL%20MOULAT.pdf\\'\\\\u001b[0m,\\\\n\",\\n       \"            \\\\u001b[32m\\'File description\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation Brief\\'\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[1m}\\\\u001b[0m\\\\n\",\\n       \"    \\\\u001b[1m]\\\\u001b[0m,\\\\n\",\\n       \"    \\\\u001b[32m\\'meta\\'\\\\u001b[0m: \\\\u001b[1m{\\\\u001b[0m\\\\n\",\\n       \"        \\\\u001b[32m\\'Title\\'\\\\u001b[0m: \\\\u001b[32m\\'EX-POST EVALUATION OF THE PROJECT:  NIGERIA: STRENGTHENING REINTEGRATION FOR RETURNEES \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32mSRARP\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m  -\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mPHASE II\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Year\\'\\\\u001b[0m: \\\\u001b[1;36m2023\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Author\\'\\\\u001b[0m: \\\\u001b[32m\\'Abderrahim El Moulat\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Best Practicesor Lessons Learnt\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Date of Publication\\'\\\\u001b[0m: \\\\u001b[32m\\'2023-05-10\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Donor\\'\\\\u001b[0m: \\\\u001b[32m\\'Government of Germany\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Brief\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Commissioner\\'\\\\u001b[0m: \\\\u001b[32m\\'Donor, IOM\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Coverage\\'\\\\u001b[0m: \\\\u001b[32m\\'Country\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Period From Date\\'\\\\u001b[0m: \\\\u001b[32m\\'nan\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Evaluation Period To Date\\'\\\\u001b[0m: \\\\u001b[32m\\'NaT\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Executive Summary\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'External Version of the Report\\'\\\\u001b[0m: \\\\u001b[32m\\'No\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Languages\\'\\\\u001b[0m: \\\\u001b[32m\\'English\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Migration Thematic Areas\\'\\\\u001b[0m: \\\\u001b[32m\\'Assistance to vulnerable migrants, Migrant training and integration \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32mincluding\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mcommunity cohesion\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m, Migration health \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32massessment, travel, health promotion, crisis-affected\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m, Return and AVRR\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Name of Project\\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32ms\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m Being Evaluated\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'Number of Pages Excluding annexes\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'Other Documents Included\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'Project Code\\'\\\\u001b[0m: \\\\u001b[32m\\'RR.0163\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Countries Covered\\'\\\\u001b[0m: \\\\u001b[1m[\\\\u001b[0m\\\\u001b[32m\\'Nigeria\\'\\\\u001b[0m\\\\u001b[1m]\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Regions Covered\\'\\\\u001b[0m: \\\\u001b[32m\\'RO Dakar\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Relevant Crosscutting Themes\\'\\\\u001b[0m: \\\\u001b[32m\\'Gender, Rights-based approach\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Report Published\\'\\\\u001b[0m: \\\\u001b[32m\\'Yes\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Terms of Reference\\'\\\\u001b[0m: \\\\u001b[32m\\'No\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Type of Evaluation Scope\\'\\\\u001b[0m: \\\\u001b[32m\\'Programme/Project\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Type of Evaluation Timing\\'\\\\u001b[0m: \\\\u001b[32m\\'Ex-post \\\\u001b[0m\\\\u001b[32m(\\\\u001b[0m\\\\u001b[32mafter the end of the project/programme\\\\u001b[0m\\\\u001b[32m)\\\\u001b[0m\\\\u001b[32m\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Type of Evaluator\\'\\\\u001b[0m: \\\\u001b[32m\\'Internal\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Level of Evaluation\\'\\\\u001b[0m: \\\\u001b[32m\\'Decentralized\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Document Subtype\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation report, Evaluation brief\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'File URL\\'\\\\u001b[0m: \\\\n\",\\n       \"\\\\u001b[32m\\'https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Internal%20Evaluation_NG20P0516_MAY_2023_F\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mINAL_Abderrahim%20EL%20MOULAT.pdf,   \\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mhttps://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/RR0163_Evaluation%20Brief_MAY_%202023_Abder\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[32mrahim%20EL%20MOULAT.pdf\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'File description\\'\\\\u001b[0m: \\\\u001b[32m\\'Evaluation Report , Evaluation Brief\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Management response\\'\\\\u001b[0m: \\\\u001b[32m\\'No\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Date added\\'\\\\u001b[0m: \\\\u001b[32m\\'Fri, 07/07/2023 - 15:35\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'Metaevaluation\\'\\\\u001b[0m: \\\\u001b[32m\\'2020-24\\'\\\\u001b[0m,\\\\n\",\\n       \"        \\\\u001b[32m\\'exclude\\'\\\\u001b[0m: nan,\\\\n\",\\n       \"        \\\\u001b[32m\\'reason\\'\\\\u001b[0m: nan\\\\n\",\\n       \"    \\\\u001b[1m}\\\\u001b[0m\\\\n\",\\n       \"\\\\u001b[1m}\\\\u001b[0m\\\\n\"\\n      ]\\n     },\\n     \"metadata\": {},\\n     \"output_type\": \"display_data\"\\n    }\\n   ],\\n   \"source\": [\\n    \"#| eval: false\\\\n\",\\n    \"fname_json = \\'../_data/output/evaluations.json\\'\\\\n\",\\n    \"evals = load_evals(fname_json)\\\\n\",\\n    \"print(evals[0])\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"To find a particular evaluation per name (e.g \\\\\"Final Evaluation of the EU-IOM Joint ...\\\\\"), we can simply:\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"metadata\": {},\\n   \"outputs\": [\\n    {\\n     \"data\": {\\n      \"text/plain\": [\\n       \"[{\\'id\\': \\'49d2fba781b6a7c0d94577479636ee6f\\',\\\\n\",\\n       \"  \\'docs\\': [{\\'Document Subtype\\': \\'Evaluation report\\',\\\\n\",\\n       \"    \\'File URL\\': \\'https://ev<TRUNCATED>"
}
```

</details>

Good! I can see the notebook has quite a bit of code. Now I understand the structure better. 

Here's what I see as the core functionality we need to extract:

1. **PDF conversion** - converting PDFs to markdown using Mistral batch API
2. **Image extraction** - saving images from PDFs  
3. **File organization** - creating folder structures for output

Before we start coding, let me ask: Would you like to start by sketching out the API design and configurable options first, or would you prefer to jump into refactoring a specific function?

Yes thinking ahead of what the api look like does not heart.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! Let's design the API together. Based on the notebook and your requirements, here's what I'm thinking:

**Main function:**
```python
ocr_pdf(pdf_path, output_dir=None, image_dir=None, model='pixtral-12b-2409', ...)
```

**Key configurable parameters:**
- `pdf_path`: Path to single PDF or folder containing PDFs
- `output_dir`: Where to save markdown files (default: `'./output'`)
- `image_dir`: Where to save extracted images (default: `'./images'`)
- `model`: Mistral model to use
- `api_key`: Mistral API key (or get from env)

**Questions for you:**

1. Should `ocr_pdf` handle both single files AND folders, or would you prefer separate functions like `ocr_pdf(file)` and `ocr_folder(folder)`?

2. What should the output structure look like? For example:
   - `output/document1.md` with images in `images/document1/img_001.png`?
   - Or something else?

3. Any other parameters you think should be configurable (e.g., batch size, retry logic, timeout)?

What are your thoughts on these?

1. I would prefer a single `ocr_pdf` function
2. something like folder_name/page1.pdf, ... and folder_name/img/ where folder_name could be either the file_name, or something custom, everything should be customizable
3. nothing for now

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! So we have:

**Single function `ocr_pdf` that handles both files and folders**

**Output structure:**
- `output_dir/folder_name/page1.md, page2.md, ...`
- `output_dir/folder_name/img/img_001.png, img_002.png, ...`
- `folder_name` defaults to PDF filename (without extension) but can be customized

**API sketch:**
```python
def ocr_pdf(
    pdf_path,           # str or Path - file or folder
    output_dir='./output',
    folder_name=None,   # if None, use PDF filename
    image_subdir='img',
    model='pixtral-12b-2409',
    api_key=None        # if None, get from env
):
    ...
```

**Question:** When `pdf_path` is a folder with multiple PDFs, should each PDF get its own subfolder? Like:
- `output_dir/pdf1/page1.md, page2.md, img/...`
- `output_dir/pdf2/page1.md, page2.md, img/...`

Or something different?

Looks fine for now.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Excellent! Now let's start building. 

Looking at the notebook, I can see there are several helper functions we'll need to extract. Should we:

1. Start with the core OCR logic (the Mistral API call)?
2. Start with PDF handling (converting PDF to images)?
3. Or start with the main `ocr_pdf` function and work our way down?

Which approach would you prefer?

Let's start now. Do you confirm you see the entire module I want to refactor? Can we plan a bit ahead how we will approach it and design the architecture? It shouldn't be that complex I think

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point - let me make sure I have the full picture. The notebook content I fetched seems truncated. Let me get the complete module so we can plan the architecture properly.

Looking at what I have so far, I can see:
- Imports and setup
- Some evaluation loading code (which we'll remove)
- But I'm missing the actual OCR functions

Can you either:
1. Share the key functions from the notebook here, or
2. Let me try to fetch more of the notebook content?

Once I see all the functions (like the Mistral API calls, PDF conversion, image handling), we can map out a clean architecture together.

In [ ]:
def notebook_to_markdown(notebook_path):
    """Convert a Jupyter notebook to markdown format"""
    import json
    
    with open(notebook_path, 'r') as f:
        nb = json.load(f)
    
    markdown_content = []
    
    for cell in nb['cells']:
        if cell['cell_type'] == 'markdown':
            markdown_content.append(''.join(cell['source']))
            markdown_content.append('\n---\n')
        elif cell['cell_type'] == 'code':
            markdown_content.append('```python')
            markdown_content.append(''.join(cell['source']))
            markdown_content.append('```\n')
    
    return '\n'.join(markdown_content)

In [ ]:
#| hide
from nbdev.showdoc import *a

In [ ]:
#| export
def foo(): pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
!nbdev_export

In [ ]:
!wget https://raw.githubusercontent.com/franckalbinet/evaluatr/refs/heads/main/nbs/03_ocr.ipynb

--2025-11-03 20:38:34--  https://raw.githubusercontent.com/franckalbinet/evaluatr/refs/heads/main/nbs/03_ocr.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 46394 (45K) [text/plain]
Saving to: ‘03_ocr.ipynb’

03_ocr.ipynb        100%[===================>]  45.31K  --.-KB/s    in 0.001s  

2025-11-03 20:38:34 (37.9 MB/s) - ‘03_ocr.ipynb’ saved [46394/46394]



In [ ]:
!mv 03_ocr.ipynb _03_ocr.ipynb